# 01 — Extração e Fusão dos Dados (FBref · Understat · Sofascore · Transfermarkt)

**Objetivo.** Construir o *dataset* bruto consolidado que sustenta a previsão da variação do valor de
mercado ($\Delta V$, $\Delta V\%$) em $T+1$ e $T+2$. Cada linha é uma observação **jogador–temporada–clube**
das *Big Five* (2019/20–2023/24, mais 2018/19 apenas para lags). Ela reúne o desempenho em $T$, os
atributos demográficos e contratuais e os valores de mercado em $T$, $T+1$ e $T+2$.

**Saída:** `data/raw/football_raw_dataset.parquet` (+ `.csv` e dicionário `_schema.json`), além das
tabelas intermediárias em `data/interim/`.

**Escopo deste notebook.** Apenas coleta, resolução de entidades e fusão. **Nenhum filtro amostral é
aplicado aqui**: o corte $p_{90}$, a minutagem mínima de 90 minutos e a exclusão de goleiros são
validados no notebook 02 e aplicados no 03. Pelo mesmo motivo, $\Delta V$ e $\Delta V\%$ não são
calculados neste ponto. São gravados os valores brutos $V_T$, $V_{T+1}$, $V_{T+2}$ e as colunas de
controle (`fb_minutes`, `is_goalkeeper`).

Toda a lógica está no pacote `football_analytics` (`src/`), coberta por testes unitários (`tests/`).
O notebook encadeia as etapas e exibe os diagnósticos que fundamentam a seção *Verificação dos
Dados* do artigo.

## Arquitetura de fontes e a remoção dos dados Opta do FBref

Em **20/01/2026** a Sports Reference retirou do FBref todas as métricas avançadas licenciadas pela
Opta (Stats Perform), **inclusive das temporadas históricas**. As páginas `passing`, `possession`,
`gca` e grande parte de `defense` continuam publicadas, mas com as células vazias. O diagnóstico da
Etapa 1 documenta isso coluna a coluna. Para preservar o vetor tático-técnico previsto nas
especificações (Tabela 4.1 e Quadro 6.1), a arquitetura passa a integrar quatro fontes:

| Fonte | Acesso | Variáveis | Papel no modelo |
|---|---|---|---|
| **FBref** | soccerdata + Chrome (Cloudflare) | minutos, jogos/titularidades, gols, assistências, chutes, cartões, faltas, cruzamentos, interceptações, desarmes ganhos, gols pró/contra e +/- em campo, ano de nascimento | Espinha dorsal da amostra (minutagem, posição, id longitudinal) |
| **Understat** | soccerdata (HTTP) | xG, npxG, xA, xGChain, xGBuildup, passes-chave | Família de gols esperados |
| **Sofascore** | API JSON via Chrome | passes certos/%, bolas longas, passes no terço final, passes-chave, grandes chances, dribles, duelos aéreos/terrestres, desarmes, cortes, recuperações, perdas de posse | Vetor tático que o FBref deixou de publicar |
| **Transfermarkt** | requests (HTML + `/ceapi`) | valor de mercado datado, data de nascimento, altura, pé, nacionalidades, posição granular, contrato atual, transferências | Baseline financeiro, alvos e controles demográficos/contratuais |

Não existe chave primária comum entre as fontes. A resolução de entidades (Etapa 5) usa o FBref
como referência e casa cada linha com as demais fontes por similaridade de nomes
(Levenshtein + Jaro-Winkler), sob restrições de clube, temporada, ano de nascimento e minutagem.

## 0. Configuração

`SMOKE_TEST = True` executa o pipeline completo num recorte mínimo (Premier League 2023/24 e
40 jogadores no Transfermarkt), gravado em `data/smoke_test/`, para validar acesso e schema antes da
coleta completa. A coleta completa leva **várias horas** a frio, quase todas no Transfermarkt
(~3 requisições por jogador). Todo recurso fica em cache em disco: interromper e reexecutar retoma
do ponto em que parou. `MAX_FETCHES_PER_RUN` limita deliberadamente as requisições por sessão.

O FBref e o Sofascore exigem uma **janela visível do Chrome** (o modo headless é reprovado pelos
anti-bots). Se não houver Chrome no sistema, instale o *Chrome for Testing* com
`.venv/bin/sbase get cft`.

In [1]:
import os
import warnings

import pandas as pd

from football_analytics.config import default_config
from football_analytics.logging_utils import setup_logging

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)

SMOKE_TEST = os.environ.get("FA_SMOKE_TEST") == "1"  # ou True: recorte mínimo em data/smoke_test/
MAX_FETCHES_PER_RUN = None  # ex.: 3000 para dividir a coleta do Transfermarkt em sessões

cfg = default_config(max_fetches_per_run=MAX_FETCHES_PER_RUN)
if SMOKE_TEST:
    cfg = cfg.as_smoke_test()
cfg.paths.ensure()

# O soccerdata lê SOCCERDATA_DIR na importação: o cache fica dentro do projeto e não no ~/soccerdata.
os.environ["SOCCERDATA_DIR"] = str(cfg.paths.soccerdata_dir)

logger = setup_logging(cfg.paths.logs / ("01_data_extraction_smoke.log" if SMOKE_TEST else "01_data_extraction.log"))
logger.info("configuração", extra=cfg.describe())
cfg.describe()

I football_analytics | configuração  root=/home/alvaro/Documents/TCC/football-analytics leagues=['ENG-Premier League', 'ESP-La Liga', 'ITA-Serie A', 'GER-Bundesliga', 'FRA-Ligue 1'] seasons=['1819', '1920', '2021', '2122', '2223', '2324'] feature_seasons=['1920', '2021', '2122', '2223', '2324'] lag_only_seasons=['1819'] fbref_pages=['standard', 'shooting', 'playing_time', 'misc'] smoke_test=False max_fetches_per_run=None browser_headless=False


{'root': '/home/alvaro/Documents/TCC/football-analytics',
 'leagues': ['ENG-Premier League',
  'ESP-La Liga',
  'ITA-Serie A',
  'GER-Bundesliga',
  'FRA-Ligue 1'],
 'seasons': ['1819', '1920', '2021', '2122', '2223', '2324'],
 'feature_seasons': ['1920', '2021', '2122', '2223', '2324'],
 'lag_only_seasons': ['1819'],
 'fbref_pages': ['standard', 'shooting', 'playing_time', 'misc'],
 'smoke_test': False,
 'max_fetches_per_run': None,
 'browser_headless': False}

In [2]:
from football_analytics import pipeline
from football_analytics.browser import find_chrome

print("Chrome:", find_chrome() or "Chrome do sistema")

/home/alvaro/Documents/TCC/football-analytics/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chrome:

/home/alvaro/Documents/TCC/football-analytics/.venv/lib/python3.12/site-packages/seleniumbase/drivers/cft_drivers/chrome-linux64/chrome

## 1. FBref — estatísticas de jogador por temporada

Uso das páginas **combinadas das Big 5** (uma requisição por página e temporada, em vez de cinco),
com rate limit de 7 s imposto pelo soccerdata. O *parser* lê cada célula pelo atributo `data-stat` e
preserva o **id do jogador e do clube no FBref**. Esses ids são estáveis entre temporadas e servem de
chave longitudinal do painel (o soccerdata os descarta). Um jogador transferido dentro das Big 5 no
meio da temporada gera uma linha por clube (`n_teams_in_season`). A agregação, se desejada, fica para
o notebook 03.

Colunas integralmente vazias são descartadas e registradas em log. Ficam só as métricas
efetivamente publicadas.

In [3]:
fb, fbref_fill = pipeline.run_fbref(cfg)

print(f"FBref: {len(fb):,} linhas jogador-temporada-clube x {fb.shape[1]} colunas")
assert set(fb["league"]) == set(cfg.league_keys), "liga ausente no FBref"
assert not fb.duplicated(["season", "fbref_player_id", "fbref_team_id"]).any()
fb.groupby(["season", "league"]).size().unstack("league")

I            fbref | temporada FBref consolidada  season=1819 rows=2656


I            fbref | temporada FBref consolidada  season=1920 rows=2732


I            fbref | temporada FBref consolidada  season=2021 rows=2822


I            fbref | temporada FBref consolidada  season=2122 rows=2922


I            fbref | temporada FBref consolidada  season=2223 rows=2889


I            fbref | temporada FBref consolidada  season=2324 rows=2852


I               io | tabela gravada  path=/home/alvaro/Documents/TCC/football-analytics/data/interim/fbref_player_season.parquet rows=16873 cols=59


I               io | tabela gravada  path=/home/alvaro/Documents/TCC/football-analytics/data/interim/fbref_opta_fill_rates.parquet rows=76 cols=4


FBref: 16,873 linhas jogador-temporada-clube x 59 colunas

league,ENG-Premier League,ESP-La Liga,FRA-Ligue 1,GER-Bundesliga,ITA-Serie A
season,,,,,
1819,508,544,561,474,569
1920,522,570,542,498,600
2021,532,582,584,505,619
2122,546,617,604,523,632
2223,569,596,606,515,603
2324,580,609,540,507,616


In [4]:
fb_metrics = sorted(c for c in fb.columns if c.startswith("fb_") and c not in
                    {"fb_player", "fb_team", "fb_position", "fb_nationality", "fb_age", "fb_birth_year"})
print(f"{len(fb_metrics)} métricas FBref publicadas:\n", ", ".join(fb_metrics))

47 métricas FBref publicadas:


fb_assists, fb_assists_per90, fb_cards_red, fb_cards_yellow, fb_cards_yellow_red, fb_crosses, fb_fouled, fb_fouls, fb_games, fb_games_complete, fb_games_starts, fb_games_subs, fb_goals, fb_goals_assists, fb_goals_assists_pens_per90, fb_goals_assists_per90, fb_goals_pens, fb_goals_pens_per90, fb_goals_per90, fb_goals_per_shot, fb_goals_per_shot_on_target, fb_interceptions, fb_minutes, fb_minutes_90s, fb_minutes_pct, fb_minutes_per_game, fb_minutes_per_start, fb_minutes_per_sub, fb_offsides, fb_on_goals_against, fb_on_goals_for, fb_own_goals, fb_pens_att, fb_pens_conceded, fb_pens_made, fb_pens_won, fb_plus_minus, fb_plus_minus_per90, fb_plus_minus_wowy, fb_points_per_game, fb_shots, fb_shots_on_target, fb_shots_on_target_pct, fb_shots_on_target_per90, fb_shots_per90, fb_tackles_won, fb_unused_subs

### 1.1 Diagnóstico: páginas avançadas do FBref após jan/2026

Taxa de preenchimento das colunas nas páginas que antes traziam os dados Opta, na temporada mais
recente do estudo. Uma taxa de 0% indica coluna publicada, mas sem valores. Este quadro documenta, na
Metodologia, por que passes, progressão, SCA/GCA e duelos vêm do Sofascore e o xG do Understat.

In [5]:
fbref_fill.pivot_table(index="column", columns="page", values="fill_rate").round(2).fillna("")

page,defense,gca,passing,possession
column,,,,
assisted_shots,,,0.0,
assists,,,1.0,
blocked_passes,0.0,,,
blocked_shots,0.0,,,
blocks,0.0,,,
...,...,...,...,...
touches_def_3rd,,,,0.0
touches_def_pen_area,,,,0.0
touches_live_ball,,,,0.0


## 2. Understat — família de gols esperados

O Understat publica xG, npxG, xA, xGChain e xGBuildup para as Big 5 desde 2014/15, por jogador e
clube. A cobertura é a mesma do FBref e o acesso é HTTP simples, sem navegador.

In [6]:
us = pipeline.run_understat(cfg)
print(f"Understat: {len(us):,} linhas")
us.groupby(["season", "league"]).size().unstack("league")

[10/19/26 09:40:04] INFO     No custom team name replacements found. You can configure these in       ]8;id=15950109;file:///home/alvaro/Documents/TCC/football-analytics/.venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=15950110;file:///home/alvaro/Documents/TCC/football-analytics/.venv/lib/python3.12/site-packages/soccerdata/_config.py#91\91]8;;\
                             /home/alvaro/Documents/TCC/football-analytics/data/raw/soccerdata/config              
                             /teamname_replacements.json.                                                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=15950116;file:///home/alvaro/Documents/TCC/football-analytics/.venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=15950117;file:///home/alvaro/Documents/TCC/football-analytics/.venv/lib/python3.12/site-packages/soccerdata/_config.py#189\189]8;;\
                             /home/alvaro/Documents/TCC/football-analytics/data/raw/soccerdata/confi               
                             g/league_dict.json.                                                                   

                    INFO     Saving cached data to                                                   ]8;id=15950124;file:///home/alvaro/Documents/TCC/football-analytics/.venv/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=15950125;file:///home/alvaro/Documents/TCC/football-analytics/.venv/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /home/alvaro/Documents/TCC/football-analytics/data/raw/soccerdata/Under               
                             stat                                                                                  

[2026-10-19 09:40:04] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: /home/alvaro/Documents/TCC/football-analytics/.venv/lib/python3.12/site-packages/tls_requests/bin/tls-client-xgo-1.13.1-linux-amd64.so


                    INFO     Successfully loaded TLS library:                                      ]8;id=15950132;file:///home/alvaro/Documents/TCC/football-analytics/.venv/lib/python3.12/site-packages/tls_requests/models/libraries.py\libraries.py]8;;\:]8;id=15950133;file:///home/alvaro/Documents/TCC/football-analytics/.venv/lib/python3.12/site-packages/tls_requests/models/libraries.py#397\397]8;;\
                             /home/alvaro/Documents/TCC/football-analytics/.venv/lib/python3.12/si                 
                             te-packages/tls_requests/bin/tls-client-xgo-1.13.1-linux-amd64.so                     

                    WARNING  /home/alvaro/Documents/TCC/football-analytics/.venv/lib/python3.12/sit ]8;id=15950140;file:///usr/lib/python3.12/warnings.py\warnings.py]8;;\:]8;id=15950141;file:///usr/lib/python3.12/warnings.py#110\110]8;;\
                             e-packages/soccerdata/_common.py:144: UserWarning: Season id "2021" is                
                             ambiguous: interpreting as "20-21"                                                    
                               warnings.warn(msg, stacklevel=1)                                                    
                                                                                                                   

I        understat | Understat lido  rows=16446 seasons=['1819', '1920', '2021', '2122', '2223', '2324']


I               io | tabela gravada  path=/home/alvaro/Documents/TCC/football-analytics/data/interim/understat_player_season.parquet rows=16446 cols=21


Understat: 16,446 linhas

league,ENG-Premier League,ESP-La Liga,FRA-Ligue 1,GER-Bundesliga,ITA-Serie A
season,,,,,
1819,505,531,548,468,543
1920,515,556,531,487,572
2021,524,570,575,496,594
2122,537,603,589,511,608
2223,554,584,585,506,577
2324,570,598,525,494,590


## 3. Sofascore — vetor tático-técnico

A API de estatísticas por torneio e temporada é consultada **por clube**
(`filters=team.in.<id>`). Cada lista tem menos de 100 jogadores, o que dispensa paginação e torna a
coleta determinística, e o grão (jogador–clube) coincide com o do FBref. São ~100 clubes por
temporada, ≈ 15 min a frio. Na temporada 2019/20 o Sofascore não publica xG, xA, recuperações nem
bloqueios. Esses campos ficam ausentes (NaN), e o xG é coberto pelo Understat.

In [7]:
ss = pipeline.run_sofascore(cfg)
print(f"Sofascore: {len(ss):,} linhas")
display(ss.groupby(["season", "league"]).size().unstack("league"))
ss.filter(like="ss_").notna().groupby(ss["season"]).mean().T.round(2)

I        sofascore | Sofascore liga-temporada  league=ENG-Premier League season=1819 teams=20


I        sofascore | Sofascore liga-temporada  league=ENG-Premier League season=1920 teams=20


I        sofascore | Sofascore liga-temporada  league=ENG-Premier League season=2021 teams=20


I        sofascore | Sofascore liga-temporada  league=ENG-Premier League season=2122 teams=20


I        sofascore | Sofascore liga-temporada  league=ENG-Premier League season=2223 teams=20


I        sofascore | Sofascore liga-temporada  league=ENG-Premier League season=2324 teams=20


... (24 outputs left out)

I               io | tabela gravada  path=/home/alvaro/Documents/TCC/football-analytics/data/interim/sofascore_player_season.parquet rows=16462 cols=50


Sofascore: 16,462 linhas

league,ENG-Premier League,ESP-La Liga,FRA-Ligue 1,GER-Bundesliga,ITA-Serie A
season,,,,,
1819,505,531,548,468,544
1920,515,557,534,487,575
2021,524,571,573,496,596
2122,538,604,591,511,609
2223,554,584,586,506,577
2324,570,598,525,494,591


season,1819,1920,2021,2122,2223,2324
ss_player,1.0,1.0,1.0,1.0,1.0,1.00
ss_team,1.0,1.0,1.0,1.0,1.0,1.00
ss_minutes_played,1.0,1.0,1.0,1.0,1.0,1.00
ss_appearances,1.0,1.0,1.0,1.0,1.0,1.00
ss_matches_started,1.0,1.0,1.0,1.0,1.0,1.00
ss_rating,1.0,1.0,1.0,1.0,1.0,1.00
ss_goals,1.0,1.0,1.0,1.0,1.0,1.00
ss_assists,1.0,1.0,1.0,1.0,1.0,1.00
ss_expected_goals,0.0,0.0,0.0,0.0,1.0,0.92
ss_expected_assists,0.0,0.0,0.0,0.0,1.0,0.99


## 4. Transfermarkt — elencos por liga-temporada

Visão detalhada do elenco (`/kader/.../plus/1`), uma página por clube e temporada (≈ 580 páginas).
Ela fornece o id do jogador no Transfermarkt, a data de nascimento (validador da resolução de
entidades), altura, pé, nacionalidades, posição granular e o valor exibido (só para conferência).

**Limitação documentada:** nas temporadas passadas o Transfermarkt substitui a coluna de contrato por
"Current club". A data de término vigente em $T$ **não é observável** nessa página. O NB01 coleta os
insumos para estimá-la no notebook 03: contrato atual, data de chegada, última renovação e histórico
de transferências (Etapa 6). A coluna `tm_joined_date_raw` reflete o estado atual do cadastro e só
pode ser usada após o filtro `joined ≤ fim de T`.

In [8]:
squads, squad_failures = pipeline.run_tm_squads(cfg)
print(f"Elencos TM: {len(squads):,} linhas | {squads['tm_player_id'].nunique():,} jogadores únicos")
if squad_failures:
    display(pd.DataFrame(squad_failures))  # reexecute a célula para tentar novamente
squads.groupby(["season", "league"])["tm_club_id"].nunique().unstack("league")

ENG-Premier League 1819:   0%|          | 0/20 [00:00<?, ?it/s]

ENG-Premier League 1819:  10%|█         | 2/20 [00:00<00:00, 18.78it/s]

ENG-Premier League 1819:  20%|██        | 4/20 [00:00<00:00, 18.71it/s]

ENG-Premier League 1819:  30%|███       | 6/20 [00:00<00:00, 19.27it/s]

ENG-Premier League 1819:  45%|████▌     | 9/20 [00:00<00:00, 20.20it/s]

ENG-Premier League 1819:  60%|██████    | 12/20 [00:00<00:00, 21.46it/s]

... (274 outputs left out)

I    transfermarkt | elencos TM  league=FRA-Ligue 1 season=2324 clubs=18


I               io | tabela gravada  path=/home/alvaro/Documents/TCC/football-analytics/data/interim/tm_squads.parquet rows=22715 cols=16


Elencos TM: 22,715 linhas | 8,828 jogadores únicos

league,ENG-Premier League,ESP-La Liga,FRA-Ligue 1,GER-Bundesliga,ITA-Serie A
season,,,,,
1819,20,20,20,18,20
1920,20,20,20,18,20
2021,20,20,20,18,20
2122,20,20,20,18,20
2223,20,20,20,18,20
2324,20,20,18,18,20


## 5. Resolução de entidades (*Entity Resolution*)

Procedimento (specs, §5.2), aplicado de forma independente ao Transfermarkt, ao Understat e ao
Sofascore:

1. **Clubes.** Dentro de cada liga-temporada, as fontes listam os mesmos 18–20 clubes. O mapeamento é
   uma atribuição 1-para-1 (algoritmo húngaro) sobre a similaridade difusa dos nomes. Aliases manuais
   (`data/external/team_aliases.csv`) têm precedência.
2. **Jogadores, bloco liga–temporada–clube.** Score composto
   $0{,}5\cdot\text{Levenshtein}_{\text{token-sort}} + 0{,}5\cdot\text{Jaro-Winkler}$, com
   $0{,}92\cdot\text{token-set}$ para nomes parciais e apelidos. A **restrição dura** de ano de
   nascimento ($|\Delta| \le 1$) desfaz homônimos no Transfermarkt. Nas fontes sem data de
   nascimento, a minutagem compatível penaliza ou valida o par. A atribuição é 1-para-1. Aceita-se
   score ≥ 85, ou ≥ 80 com validador forte. Diminutivos são tratados por uma regra explícita:
   sobrenome idêntico e mesma inicial do prenome valem 88 ("Max" ↔ "Maximilian Kilman",
   "Gio" ↔ "Giovanni Reyna"). O último par restante de um bloco é aceito se validado e com score
   ≥ 60 ("Rodri" ↔ "Rodrigo Hernández").
3. **Fallback na liga-temporada** para empréstimos e inscrições tardias: score ≥ 90, com validador
   forte obrigatório.
4. **Propagação pelo id do FBref.** O id associado ao mesmo jogador em outra temporada é reutilizado.
5. **Agregado da temporada** (Understat e Sofascore). Essas fontes publicam uma única linha para o
   jogador transferido no meio da temporada, enquanto o FBref publica uma por clube. A minutagem é
   validada também contra o total do jogador na temporada. A linha agregada é compartilhada pelas
   linhas do FBref e marcada em `us_is_season_aggregate` / `ss_is_season_aggregate`, para que o
   notebook 03 agregue o FBref antes de usar essas métricas e não conte em dobro.
6. **Temporadas vizinhas** (só Transfermarkt): ano de nascimento idêntico e score ≥ 92.
7. **Conflitos.** Um id FBref ligado a mais de um id Transfermarkt é resolvido pela maioria
   ponderada por minutos e marcado (`tm_id_conflict`). Overrides manuais prevalecem.

In [9]:
matches = pipeline.run_entity_resolution(cfg, fb, squads, us, ss)

tm_maps = matches.team_maps
print("Mapeamento de clubes por método:")
display(tm_maps.groupby(["source", "method"]).size().unstack(fill_value=0))
print("Pares de clube com menor score (conferir; corrigir via team_aliases.csv):")
tm_maps[tm_maps["method"] == "assignment"].nsmallest(12, "score")

W entity_resolution | mapeamento de clube com baixa confiança — confira/adicione alias  league=GER-Bundesliga season=1819 team=Hertha BSC target=Hertha Berlin score=71.2


W entity_resolution | mapeamento de clube com baixa confiança — confira/adicione alias  league=GER-Bundesliga season=1920 team=Hertha BSC target=Hertha Berlin score=71.2


W entity_resolution | mapeamento de clube com baixa confiança — confira/adicione alias  league=GER-Bundesliga season=2021 team=Hertha BSC target=Hertha Berlin score=71.2


W entity_resolution | mapeamento de clube com baixa confiança — confira/adicione alias  league=GER-Bundesliga season=2122 team=Hertha BSC target=Hertha Berlin score=71.2


W entity_resolution | mapeamento de clube com baixa confiança — confira/adicione alias  league=GER-Bundesliga season=2223 team=Hertha BSC target=Hertha Berlin score=71.2


W entity_resolution | número de clubes difere entre fontes  league=FRA-Ligue 1 season=1819 left=20 right=22


... (15 outputs left out)

Mapeamento de clubes por método:

method,alias,assignment
source,,
sofascore,0,586
transfermarkt,42,544
understat,5,581


Pares de clube com menor score (conferir; corrigir via team_aliases.csv):

,league,season,team,target_team,score,method,source
957,GER-Bundesliga,1819,Hertha BSC,Hertha Berlin,71.250000,assignment,understat
974,GER-Bundesliga,1920,Hertha BSC,Hertha Berlin,71.250000,assignment,understat
990,GER-Bundesliga,2021,Hertha BSC,Hertha Berlin,71.250000,assignment,understat
1007,GER-Bundesliga,2122,Hertha BSC,Hertha Berlin,71.250000,assignment,understat
1032,GER-Bundesliga,2223,Hertha BSC,Hertha Berlin,71.250000,assignment,understat
1426,FRA-Ligue 1,1819,Rennes,Stade Rennais,76.923077,assignment,sofascore
1451,FRA-Ligue 1,1920,Rennes,Stade Rennais,76.923077,assignment,sofascore
1457,FRA-Ligue 1,2021,Rennes,Stade Rennais,76.923077,assignment,sofascore
1476,FRA-Ligue 1,2122,Rennes,Stade Rennais,76.923077,assignment,sofascore
1497,FRA-Ligue 1,2223,Rennes,Stade Rennais,76.923077,assignment,sofascore


In [10]:
reports = {
    "transfermarkt": pipeline.match_report(fb, matches.tm, "tm_player_id"),
    "understat": pipeline.match_report(fb, matches.understat, "understat_player_id"),
    "sofascore": pipeline.match_report(fb, matches.sofascore, "sofascore_player_id"),
}
summary = pd.concat({k: v.set_index(["league", "season"])[["target_rows", "target_match_rate"]]
                     for k, v in reports.items()}, axis=1)
print("Taxa de match no universo-alvo (jogadores de linha com ≥ 90 min) — meta ≥ 97% no Transfermarkt")
summary.round(3)

Taxa de match no universo-alvo (jogadores de linha com ≥ 90 min) — meta ≥ 97% no Transfermarkt

transfermarkt                     understat                     sofascore                  
                            target_rows target_match_rate target_rows target_match_rate target_rows target_match_rate
league             season                                                                                            
ENG-Premier League 1819             421             1.000         421             0.993         421             0.998
                   1920             436             1.000         436             0.998         436             1.000
                   2021             435             1.000         435             0.998         435             1.000
                   2122             447             1.000         447             0.996         447             0.996
                   2223             457             1.000         457             0.998         457             0.998
                   2324             456             1.000         456             0.998         456             0.998
ESP-La Liga        1819             459             1.000         459             0.996         459             0.996
                   1920             466             1.000         466             0.987         466             0.989
                   2021             478             1.000         478             0.994         478             0.990
                   2122             503             1.000         503             0.990         503             0.988
                   2223             478             0.998         478             0.998         478             0.992
                   2324             486             0.998         486             0.992         486             0.990
FRA-Ligue 1        1819             460             1.000         460             1.000         460             0.998
                   1920             432             1.000         432             0.993         432             1.000
                   2021             463             0.998         463             0.996         463             0.994
                   2122             473             1.000         473             0.994         473             0.994
                   2223             487             1.000         487             0.990         487             0.996
                   2324             433             0.998         433             0.988         433             0.995
GER-Bundesliga     1819             394             1.000         394             1.000         394             1.000
                   1920             414             0.998         414             0.993         414             0.995
                   2021             412             1.000         412             0.995         412             1.000
                   2122             410             1.000         410             0.993         410             1.000
                   2223             406             1.000         406             0.993         406             0.998
                   2324             403             1.000         403             0.990         403             0.998
ITA-Serie A        1819             466             1.000         466             1.000         466             0.998
                   1920             483             1.000         483             0.998         483             0.998
                   2021             481             1.000         481             0.996         481             0.996
                   2122             489             1.000         489             0.994         489             0.996
                   2223             486             1.000         486             0.996         486             0.996
                   2324             485             1.000         485             0.990         485             0.992

In [11]:
for name, m in [("transfermarkt", matches.tm), ("understat", matches.understat), ("sofascore", matches.sofascore)]:
    print(f"{name:14s}", m["match_method"].value_counts().to_dict())

worst = (fb.loc[matches.tm["left_index"], ["season", "league", "fb_team", "fb_player", "fb_birth_year"]]
         .assign(method=matches.tm["match_method"].to_numpy(), score=matches.tm["match_score"].to_numpy(),
                 tm_player_id=matches.tm["tm_player_id"].to_numpy()))
squad_names = squads.drop_duplicates("tm_player_id").set_index("tm_player_id")["tm_player"]
worst["tm_player"] = worst["tm_player_id"].map(squad_names)
print("\nMatches aceitos com menor score (amostra para auditoria manual):")
display(worst[worst["method"].isin(["fuzzy", "fuzzy_validated", "residual", "league_fallback", "cross_season"])]
        .nsmallest(30, "score"))

transfermarkt 

{'exact': 15857, 'fuzzy': 859, 'manual': 114, 'unmatched': 32, 'fuzzy_validated': 9, 'league_fallback': 2}

understat     

{'exact': 15067, 'fuzzy': 1166, 'season_aggregate': 427, 'unmatched': 114, 'propagated': 50, 'residual': 39, 'fuzzy_validated': 10}

sofascore     

{'exact': 15429, 'fuzzy': 788, 'season_aggregate': 425, 'unmatched': 91, 'league_fallback': 80, 'propagated': 27, 'residual': 24, 'fuzzy_validated': 9}


Matches aceitos com menor score (amostra para auditoria manual):

,season,league,fb_team,fb_player,fb_birth_year,method,score,tm_player_id,tm_player
11183,2223,ESP-La Liga,Valladolid,Babatunde Akinsola,2003.0,fuzzy_validated,81.712967,924447.0,Tunde Akinsola
16579,2324,ESP-La Liga,Almería,Marciano Tchami,2004.0,fuzzy_validated,82.212418,1013774.0,Marciano Sanca
10648,2122,ESP-La Liga,Sevilla,Valentino Scotta,2001.0,fuzzy_validated,82.634644,537373.0,Valentino Fattore
2754,1920,ENG-Premier League,Chelsea,Faustino Anjorin,2001.0,fuzzy_validated,84.523804,433181.0,Tino Anjorin
7113,2021,ESP-La Liga,Elche,Omenuke Mfulu,1994.0,fuzzy_validated,84.632111,203516.0,Nuke Mfulu
15760,2324,ESP-La Liga,Las Palmas,Omenuke Mfulu,1994.0,fuzzy_validated,84.632111,203516.0,Nuke Mfulu
764,1819,ENG-Premier League,Fulham,Fabricio,1987.0,fuzzy_validated,84.711540,45882.0,Fabri
3456,1920,ESP-La Liga,Mallorca,Fabricio,1987.0,fuzzy_validated,84.711540,45882.0,Fabri
10319,2122,ESP-La Liga,Athletic Club,Alex Petxa,1997.0,fuzzy_validated,84.711540,527838.0,Álex Petxarroman
6172,2021,ITA-Serie A,Genoa,Steeve-Mike Ebongue,2001.0,fuzzy,87.225479,702372.0,Steeve-Mike Eyango


In [12]:
unmatched = (fb.loc[matches.tm.loc[matches.tm["tm_player_id"].isna(), "left_index"]]
             .query("~is_goalkeeper")
             .sort_values("fb_minutes", ascending=False))
print(f"Linhas sem Transfermarkt: {len(unmatched):,} (exibindo as de maior minutagem)")
pipeline.save_table(unmatched, pipeline.interim(cfg, "unmatched_fbref_tm"), source="revisão manual")
unmatched[["season", "league", "fb_team", "fb_player", "fb_birth_year", "fb_minutes", "fbref_player_id"]].head(25)

Linhas sem Transfermarkt: 24 (exibindo as de maior minutagem)

I               io | tabela gravada  path=/home/alvaro/Documents/TCC/football-analytics/data/interim/unmatched_fbref_tm.parquet rows=24 cols=59


,season,league,fb_team,fb_player,fb_birth_year,fb_minutes,fbref_player_id
11636,2223,ESP-La Liga,Almería,Álex Centelles,1997.0,1123.0,ae852b7d
14517,2324,ESP-La Liga,Almería,Álex Centelles,1997.0,980.0,ae852b7d
6402,2021,FRA-Ligue 1,Metz,Vagner Gonçalves,1996.0,967.0,743fd71f
2737,1920,GER-Bundesliga,Düsseldorf,Opoku Ampomah,1996.0,500.0,e7199efb
15386,2324,FRA-Ligue 1,Metz,N'Guessan Kouadio,2003.0,110.0,eef3f9c6
9231,2122,FRA-Ligue 1,Metz,Vagner Gonçalves,1996.0,86.0,743fd71f
8256,2122,ESP-La Liga,Getafe,Amankwaa Akurugu,2001.0,79.0,69c5633e
5436,2021,ESP-La Liga,Getafe,Amankwaa Akurugu,2001.0,45.0,69c5633e
12392,2223,ESP-La Liga,Sevilla,Diego Iturralde,2003.0,45.0,9d33c71a
14343,2324,ENG-Premier League,Brighton,Benicio Boaitey,2004.0,40.0,4c5835e6


## 6. Transfermarkt — dados por jogador

Somente para os jogadores casados (todas as temporadas), em três endpoints com cache por jogador:

* **Histórico de valor de mercado** (`/ceapi/marketValueDevelopment/graph/{id}`): todas as
  avaliações datadas, com o clube na data. Insumo de $V_T$, $V_{T+1}$ e $V_{T+2}$.
* **Perfil** (`/profil/spieler/{id}`): contrato **atual** (término, opção, empréstimo), data de
  chegada ao clube atual e última renovação.
* **Histórico de transferências** (`/ceapi/transferHistory/list/{id}`): datas, origem, destino,
  empréstimos e valores. Insumo, junto com o perfil, da estimativa de `Tempo_Contrato` em $T$ e do
  dummy *Bosman* (notebook 03).

Falhas pontuais vão para `failed_fetches.json` e são re-tentadas na execução seguinte. Um bloqueio
(403/429 persistentes) interrompe a coleta de forma limpa, para retomada posterior.

In [13]:
players = pipeline.players_to_fetch(cfg, matches.tm, squads)
print(f"{len(players):,} jogadores Transfermarkt a consultar")
tm_players = pipeline.run_tm_players(cfg, players)
{k: v.shape for k, v in tm_players.items()}

5,999 jogadores Transfermarkt a consultar

I    transfermarkt | coleta por jogador  kind=market_values players=5999 to_fetch=44 retrying=1


market_values:   0%|          | 0/5999 [00:00<?, ?it/s]

market_values:   3%|▎         | 152/5999 [00:00<00:03, 1516.62it/s]

market_values:   5%|▌         | 304/5999 [00:00<00:03, 1488.13it/s]

market_values:   8%|▊         | 453/5999 [00:00<00:03, 1482.94it/s]

... (170 outputs left out)

I               io | tabela gravada  path=/home/alvaro/Documents/TCC/football-analytics/data/interim/tm_market_values_long.parquet rows=156824 cols=5


I               io | tabela gravada  path=/home/alvaro/Documents/TCC/football-analytics/data/interim/tm_profiles.parquet rows=5955 cols=14


I               io | tabela gravada  path=/home/alvaro/Documents/TCC/football-analytics/data/interim/tm_transfers_long.parquet rows=67606 cols=13


{'market_values': (156824, 5),
 'profiles': (5955, 14),
 'transfers': (67606, 13)}

## 7. Fusão e *dataset* final

A base é o FBref. Sobre ela entram, por *left join* na linha casada: atributos do Transfermarkt
(estáticos de qualquer temporada; sazonais, como clube e posição, só da mesma temporada), perfil
atual, Understat e Sofascore. Os prefixos identificam a fonte: `fb_`, `us_`, `ss_`, `tm_`,
`tm_profile_`, `tm_value_`.

**Valor de mercado em $T+k$.** É o **último valor publicado até 30/jun do fim da temporada
$T+k$**, com validade máxima de 183 dias. A escolha evita *look-ahead*: $V_T$ nunca incorpora
informação posterior ao fim da temporada de referência. Avaliações velhas demais (jogador sem
atualização ou aposentado) ficam ausentes em vez de serem tratadas como vigentes. São gravados
também a data, o clube na data e a defasagem em dias.

In [14]:
dataset = pipeline.build_dataset(cfg, fb, squads, us, ss, matches, tm_players)
pipeline.save_dataset(cfg, dataset)

key = ["season", "fbref_player_id", "fbref_team_id"]
assert not dataset.duplicated(key).any()
assert len(dataset) == len(fb), "a fusão não pode criar nem perder linhas do FBref"
print(f"Dataset final: {len(dataset):,} linhas x {dataset.shape[1]} colunas -> {cfg.paths.final_dataset}")
dataset.head(3).T.head(60)

I               io | tabela gravada  path=/home/alvaro/Documents/TCC/football-analytics/data/raw/football_raw_dataset.parquet rows=16873 cols=171


I         pipeline | dataset final gravado  path=/home/alvaro/Documents/TCC/football-analytics/data/raw/football_raw_dataset.parquet rows=16873


Dataset final: 16,873 linhas x 171 colunas -> /home/alvaro/Documents/TCC/football-analytics/data/raw/football_raw_dataset.parquet

,0,1,2
season,1819,1819,1819
league,ENG-Premier League,ENG-Premier League,ENG-Premier League
is_lag_only,True,True,True
fbref_player_id,ef619b0b,e3a5814e,6ca5ec4b
fb_player,Aaron Ramsey,Ainsley Maitland-Niles,Alex Iwobi
fbref_team_id,18bb7c10,18bb7c10,18bb7c10
fb_team,Arsenal,Arsenal,Arsenal
fb_position,MF,"MF,DF","MF,FW"
is_goalkeeper,False,False,False
fb_birth_year,1990.0,1997.0,1996.0


## 8. Diagnóstico de cobertura e limitações

Cobertura no universo-alvo preliminar: jogadores de linha com ≥ 90 min, nas temporadas de referência.
O corte $p_{90}$ e a validação formal ficam no notebook 02.

In [15]:
cov = pipeline.coverage(dataset)
cov.style.format({c: "{:.1%}" for c in cov.columns if c not in ("league", "season", "rows")})

,league,season,rows,tm_matched,understat_matched,sofascore_matched,value_T,value_T1,value_T2
0,ENG-Premier League,1920,436,100.0%,99.8%,100.0%,97.9%,97.7%,96.8%
1,ENG-Premier League,2021,435,100.0%,99.8%,100.0%,99.8%,97.9%,96.3%
2,ENG-Premier League,2122,447,100.0%,99.6%,99.6%,98.9%,97.1%,96.0%
3,ENG-Premier League,2223,457,100.0%,99.8%,99.8%,98.9%,99.3%,97.8%
4,ENG-Premier League,2324,456,100.0%,99.8%,99.8%,99.6%,98.5%,97.4%
5,ESP-La Liga,1920,466,100.0%,98.7%,98.9%,96.6%,96.4%,95.7%
6,ESP-La Liga,2021,478,100.0%,99.4%,99.0%,98.7%,97.7%,94.8%
7,ESP-La Liga,2122,503,100.0%,99.0%,98.8%,98.0%,96.0%,94.6%
8,ESP-La Liga,2223,478,99.8%,99.8%,99.2%,98.3%,96.9%,94.1%
9,ESP-La Liga,2324,486,99.8%,99.2%,99.0%,97.7%,96.9%,94.9%


In [16]:
target = dataset[~dataset["is_lag_only"]]
print("Linhas por condição (informativo — filtros aplicados nos notebooks 02/03):")
print(f"  total (temporadas de referência)  {len(target):>7,}")
print(f"  goleiros                          {int(target['is_goalkeeper'].sum()):>7,}")
print(f"  < 90 minutos                      {int((target['fb_minutes'].fillna(0) < 90).sum()):>7,}")
print(f"  linha, >= 90 min, com V_T         {int(((~target['is_goalkeeper']) & (target['fb_minutes'] >= 90) & target['tm_value_T'].notna()).sum()):>7,}")

Linhas por condição (informativo — filtros aplicados nos notebooks 02/03):

  total (temporadas de referência)   14,217

  goleiros                            1,041

  < 90 minutos                        1,836

  linha, >= 90 min, com V_T          11,256

### Limitações a registrar na Metodologia

1. **Remoção dos dados Opta do FBref (jan/2026).** O vetor tático foi recomposto com Sofascore
   (passes, duelos, dribles, ações defensivas) e Understat (xG/xA). As fontes têm modelos de evento
   e definições próprias. Por exemplo, o xG do Understat não é o xG Opta, e os "duelos" do Sofascore
   não são os "aerials" do FBref.
2. **Sofascore 2019/20 sem xG, xA, recuperações e bloqueios.** A família xG vem do Understat em
   todas as temporadas.
3. **Contrato histórico não observável.** O Transfermarkt só publica o contrato atual. A duração
   restante em $T$ é estimada no notebook 03 a partir de chegada, renovações e transferências, com
   flag de estimativa.
4. **Clean sheets com o atleta em campo** não são publicados por jogador. A aproximação disponível
   são os gols sofridos com o atleta em campo (`fb_on_goals_against`).
5. **Convocação para seleção e índice de reputação** (specs, §4.2) não são coletados neste notebook
   e ficam como trabalho futuro.
6. **Resolução de entidades.** Os pares de menor score e os não casados de maior minutagem estão em
   `data/interim/` para auditoria manual (`manual_player_overrides.csv`).